# Point Cloud to CAD-sequence

In this notebook the complete interactive pipeline for encoding point clouds into a latent space, from which DeepCAD decodes a CAD-sequence.

In [84]:
import os
import sys
import shutil
import glob
import json
import argparse
import importlib

import numpy as np
import pandas as pd
pd.set_option('display.max_rows', None)
import h5py
import torch
import torch.nn.functional as F

from OCC.Core.BRepCheck import BRepCheck_Analyzer
from OCC.Extend.DataExchange import read_step_file, write_step_file
from OCC.Core.STEPControl import STEPControl_Reader
from OCC.Core.StlAPI import StlAPI_Writer
from OCC.Core.BRepMesh import BRepMesh_IncrementalMesh

sys.path.append("..")
sys.path.append("../code")

from dataset import PointCloudEmbeddingSequenceDataset
from models.DeepCAD.config.configAE import ConfigAE
from models.DeepCAD.trainer.trainerAE import TrainerAE
from models.DeepCAD.cadlib.extrude import CADSequence
from models.DeepCAD.cadlib.visualize import vec2CADsolid, create_CAD
from models.DeepCAD.utils.file_utils import ensure_dir

### PointNet++

In [2]:
def inplace_relu(m):
    classname = m.__class__.__name__
    if classname.find('ReLU') != -1:
        m.inplace=True

def load_pointnet():
    sys.path.append(os.path.join('..', 'models','Pointnet_Pointnet2_pytorch', 'models'))
    model = importlib.import_module('pointnet2_cls_ssg')
    classifier = model.get_model(latent_dim, normal_channel=False)
    criterion = model.get_loss_mse()
    classifier.apply(inplace_relu) 
    
    saved_model = torch.load(model_path, map_location=torch.device('cpu'), weights_only=True)
    state_dict = saved_model['model_state_dict']
    if 'module.' in next(iter(state_dict)):
        monitor.log_and_print("Model was saved wrapped in nn.DataParallel.\nRemoving 'module.' from state dict.")
        state_dict = {k.replace('module.', ''): v for k, v in state_dict.items()}#
    classifier.eval()
    classifier.load_state_dict(state_dict)
    print(f"Loading PointNet++ from {os.path.abspath(model_path)}")
    return classifier

### DeepCAD

In [3]:
def load_deepcad(cfg):
    tr_agent = TrainerAE(cfg)
    tr_agent.load_ckpt(cfg.ckpt)
    tr_agent.net.eval()
    return tr_agent

### Load data

In [4]:
def get_data(indices, dataset):
    pc_list = []
    lat_rep_list = []  
    pc_paths = []
    cad_seq_list = []
    pc_dir = os.path.join(results_dir, "infered_point_clouds")
    if os.path.exists(pc_dir):
        shutil.rmtree(pc_dir)
    os.mkdir(pc_dir)
    
    for i in indices:
        pc, lat_rep, cad_seq = dataset[i]
        pc_path = os.path.abspath(dataset.get_pc_path(i))
        pc_path_destination = os.path.join(pc_dir, os.path.basename(pc_path))
        shutil.copy2(pc_path, pc_path_destination)
        pc_paths.append(pc_path_destination)
        pc_list.append(pc)
        lat_rep_list.append(lat_rep)
        cad_seq_list.append(cad_seq)

    with h5py.File(h5_file, 'a') as hf:
        dt = h5py.special_dtype(vlen=str)
        path_dataset = hf.create_dataset("pc_paths", shape=(len(pc_paths),), dtype=dt)
        path_dataset[:] = pc_paths
        
    pc_batch = torch.stack(pc_list, dim=0)
    lat_rep_batch = torch.stack(lat_rep_list, dim=0)
    cad_seq_batch = torch.stack(cad_seq_list, dim=0)
    
    return pc_batch, lat_rep_batch, cad_seq_batch

### Inference

In [5]:
def infer_pointnet(indices, dataset, model):
    with h5py.File(h5_file, 'w') as hf:
        z_pred = hf.create_dataset('z_pred', 
                                   shape=(len(indices), latent_dim), 
                                   dtype=np.float32)
        z_target = hf.create_dataset('z_target',
                                     shape=(len(indices), latent_dim),
                                     dtype = np.float32)
        seq_target = hf.create_dataset('seq_target', 
                                       shape=(len(indices), cfg.max_total_len, cfg.n_args + 1), 
                                       dtype=np.int64)
        
        pc, lat_rep, cad_seq = get_data(indices, dataset)
        z_target[:] = lat_rep
        seq_target[:] = cad_seq

        criterion_loader = importlib.import_module('pointnet2_cls_ssg')
        criterion = criterion_loader.get_loss_mse()
        
        with torch.no_grad():
            pc = pc.transpose(2, 1)
            pred, _ = model(pc)
            z_pred[:] = pred.detach()
            loss = criterion(pred, lat_rep)
            print(f"Avg. MSE-Loss: {loss.detach().item():.8e}")
            return pred, cad_seq

In [6]:
def infer_deepcad(pred, cad_seq, tr_agent):
    with h5py.File(h5_file, 'a') as hf:
        seq_pred = hf.create_dataset('seq_pred', 
                                     shape=(pred.shape[0], cfg.max_total_len, cfg.n_args + 1), 
                                     dtype=np.int64)
        cmd_logits = hf.create_dataset('cmd_logits', 
                                       shape=(pred.shape[0], cfg.max_total_len, cfg.n_commands), 
                                       dtype=np.float32)
        args_logits = hf.create_dataset('args_logits', 
                                       shape=(pred.shape[0], cfg.max_total_len, cfg.n_args, cfg.args_dim + 1), 
                                       dtype=np.float32)
        with torch.no_grad():
            pred = pred.unsqueeze(dim = 1)
            output = tr_agent.decode(pred)

            output["tgt_commands"] = cad_seq[:, :, 0] 
            output["tgt_args"] = cad_seq[:, :, 1:]
            loss_dict = tr_agent.loss_func(output)
            
            batch_out_vec = tr_agent.logits2vec(output)
            
            cmd_logits[:] = output['command_logits']
            args_logits[:] = output['args_logits']
            seq_pred[:] = batch_out_vec
            
            print(f"Avg. Command-Loss: {loss_dict['loss_cmd'].detach().cpu().item():.8e}")
            print(f"Avg. Argument-Loss: {loss_dict['loss_args'].detach().cpu().item():.8e}")

### Utils

In [7]:
def softmax(x):
    e_x = np.exp(x - np.max(x))
    return e_x / e_x.sum(axis=-1, keepdims=True)

In [29]:
def cross_entropy(logits, target):
    logits = torch.from_numpy(logits).unsqueeze(0)
    target = torch.tensor([target]).long()
    #print(logits.shape, target.shape)
    return F.cross_entropy(logits, target)

### Visualization

In [42]:
def show_results(idx):
    with h5py.File(h5_file, "r") as hf:
        pc_path = hf['pc_paths'][idx].decode("utf-8")
        args_logits = hf['args_logits'][idx]
        cmd_logits = hf['cmd_logits'][idx]
        seq_pred = hf['seq_pred'][idx]
        seq_target = hf['seq_target'][idx]
        z_pred = hf['z_pred'][idx]
        z_target = hf['z_target'][idx]

    ALL_COMMANDS = ['Line', 'Arc', 'Circle', 'EOS', 'SOL', 'Ext']
    LINE_IDX = ALL_COMMANDS.index('Line')
    ARC_IDX = ALL_COMMANDS.index('Arc')
    CIRCLE_IDX = ALL_COMMANDS.index('Circle')
    EOS_IDX = ALL_COMMANDS.index('EOS')
    SOL_IDX = ALL_COMMANDS.index('SOL')
    EXT_IDX = ALL_COMMANDS.index('Ext')

    print(f"Point Cloud path: {pc_path}")
   # print(args_logits.shape, cmd_logits.shape, seq_pred.shape, seq_target.shape, z_pred.shape, z_target.shape)
    for idx, command in enumerate(ALL_COMMANDS):
        print(f"{idx} -> {command}")

    target_commands = []
    predicted_commands = []
    pred_commands_prob = []
    cmd_loss = []
    cmd_loss_torch = []
    target_commands_prob = []
    
    all_pred_commands = list(seq_pred[:, 0])
    predicted_seq_length = list(seq_target[:, 0]).index(EOS_IDX) + 3
    target_seq_length = list(seq_pred[:, 0]).index(EOS_IDX) + 3
    seq_length = target_seq_length if target_seq_length >= predicted_seq_length else predicted_seq_length
    # print(cmd_logits[:seq_length, :])
    cmd_logits_softmax = softmax(cmd_logits[:seq_length, :])
    
    for i in range(seq_length):
        predicted_commands.append(int(all_pred_commands[i]))
        target_commands.append(int(seq_target[i, 0]))
        pred_commands_prob.append(round(float(cmd_logits_softmax[i, predicted_commands[i]]) * 100, 5))
        cmd_loss.append(cross_entropy(cmd_logits[i,:], target_commands[i]).item())
        target_commands_prob.append(round(float(cmd_logits_softmax[i, target_commands[i]]) * 100, 5))
    
    df = pd.DataFrame(list(zip(target_commands, predicted_commands, target_commands_prob, pred_commands_prob, cmd_loss)),
                      columns=['trgt', 'pred','prob_trgt', 'prob_pred', 'loss'])
    print(f"Sum CADLoss for {seq_length} commands:  {sum(cmd_loss):.8e}")
    print(f"Mean CADLoss for {seq_length} commands: {np.mean(cmd_loss):.8e}")
    return df

### Export to .step, .stl and .obj

In [10]:
def export2step():
    form = "h5"
    filter = True
    output_dir = os.path.join(results_dir, "step_files")
    if os.path.exists(output_dir):
        shutil.rmtree(output_dir)
    os.mkdir(output_dir)
    h5_path = os.path.join(results_dir, "data.h5")

    with h5py.File(h5_path, 'r') as fp:
        out_vec = fp['seq_pred'][:].astype(np.float64)
        names = fp['pc_paths'][:]
        print(out_vec.shape)
        for i, seq in enumerate(out_vec):
            pc_path = names[i].decode('utf-8')
            out_shape = vec2CADsolid(seq)
    
            if filter:
                analyzer = BRepCheck_Analyzer(out_shape)
                if not analyzer.IsValid():
                    print(f"CAD-sequence of {os.path.basename(pc_path)} is invalid.")
                    continue
    
            pc_name = os.path.splitext(os.path.basename(pc_path))[0]
            save_path = os.path.join(output_dir, pc_name + ".step")
            write_step_file(out_shape, save_path)


In [11]:
def step2stl():
    step_files = glob.glob(os.path.join(results_dir, "step_files", "*.step"))
    obj_dir = os.path.join(results_dir, "stl_files")
    if os.path.exists(obj_dir):
        shutil.rmtree(obj_dir)
    os.mkdir(obj_dir)

    for step_file in step_files:

        save_path = os.path.join(obj_dir, os.path.splitext(os.path.basename(step_file))[0] + ".stl")
    
        step_reader = STEPControl_Reader()
        step_reader.ReadFile(step_file)
        step_reader.TransferRoots()
        shape = step_reader.OneShape()

        BRepMesh_IncrementalMesh(shape, 0.5)

        stl_writer = StlAPI_Writer()
        stl_writer.Write(shape, save_path)

In [12]:
def step2obj(): # .obj files from this function lead to malformed file error in cloud compare
    step_files = glob.glob(os.path.join(results_dir, "step_files", "*.step"))
    obj_dir = os.path.join(results_dir, "obj_files")
    if os.path.exists(obj_dir):
        shutil.rmtree(obj_dir)
    os.mkdir(obj_dir)
    
    for step_file in step_files:
        shape = read_step_file(step_file)
        save_path = os.path.join(obj_dir, os.path.splitext(os.path.basename(step_file))[0] + ".obj")
        write_step_file(shape, save_path)

## Start

### Variables

Store the models in ```experiments```, a results directory will be created for each respective model.

In [13]:
model_name = "best_5"

In [14]:
model_path = os.path.join("experiments", model_name) + ".pth"
results_dir = os.path.join("experiments", model_name + "_results")
if not os.path.exists(results_dir):
    os.mkdir(results_dir)
h5_file = os.path.join(results_dir, "data.h5")
cfg = ConfigAE('test', model_path="../data/latent")
latent_dim = 256

In [15]:
pointnet_plusplus = load_pointnet()
deepcad = load_deepcad(cfg)

Loading PointNet++ from /Users/saidharb/Documents/LocalDocuments/Master-Thesis/Point-Cloud-Reconstruction/notebooks/experiments/best_5.pth
Loading checkpoint from /Users/saidharb/Documents/LocalDocuments/Master-Thesis/Point-Cloud-Reconstruction/data/latent/pretrained/model/ckpt_epoch1000.pth ...


In [16]:
dataset = PointCloudEmbeddingSequenceDataset("../data", 'test')
print(f"Dataset contains {len(dataset)} samples.")

Dataset contains 8038 samples.


In [22]:
indices = [1]

In [23]:
pred, trgt_cad_seq = infer_pointnet(indices, dataset, pointnet_plusplus)

Avg. MSE-Loss: 9.63546336e-02


In [24]:
infer_deepcad(pred, trgt_cad_seq, deepcad)

Avg. Command-Loss: 2.13566160e+00
Avg. Argument-Loss: 2.34299603e+01


In [43]:
show_results(0)

Point Cloud path: experiments/best_5_results/infered_point_clouds/00440420.ply
0 -> Line
1 -> Arc
2 -> Circle
3 -> EOS
4 -> SOL
5 -> Ext
Sum CADLoss for 13 commands:  2.77636014e+01
Mean CADLoss for 13 commands: 2.13566165e+00


,trgt,pred,prob_trgt,prob_pred,loss
0,4,4,99.99920,99.99920,7.986991e-06
1,0,0,99.99999,99.99999,1.192093e-07
2,0,0,100.00000,100.00000,0.000000e+00
3,0,0,99.99995,99.99995,4.768370e-07
4,0,0,100.00000,100.00000,0.000000e+00
5,0,0,99.99962,99.99962,3.695481e-06
6,0,0,99.99517,99.99517,4.827860e-05
7,0,4,0.00000,99.99985,1.813169e+01
8,0,2,0.00656,99.97418,9.631772e+00
9,5,5,99.99222,99.99222,7.784064e-05


Observation:

TODO: 
- check calculation of loss
- refactor this markdown
- arg bezeichnungen einfügen?
- refactor code
- 

- If arg 223 is the target, in the logits it has the index 224, because we shift every index by +1 because of the -1 pad value
- Otherwise the -1 pad value would lead to choosing the last element

-     args_softmax is a tensor of shape (seq_length, 16, 257), when indexing it with three tensors, the first two dimension are handled as a flattened grid. Therefore the indexing tensor in the first two dimension have to have a shape of seq_length * 16

In [154]:
def show_results_args(sample_idx, cmd_idx):
    with h5py.File(h5_file, "r") as hf:
        pc_path = hf['pc_paths'][sample_idx].decode("utf-8")
        args_logits = hf['args_logits'][sample_idx]
        cmd_logits = hf['cmd_logits'][sample_idx]
        seq_pred = hf['seq_pred'][sample_idx]
        seq_target = hf['seq_target'][sample_idx]
        z_pred = hf['z_pred'][sample_idx]
        z_target = hf['z_target'][sample_idx]

    ALL_COMMANDS = ['Line', 'Arc', 'Circle', 'EOS', 'SOL', 'Ext']
    LINE_IDX = ALL_COMMANDS.index('Line')
    ARC_IDX = ALL_COMMANDS.index('Arc')
    CIRCLE_IDX = ALL_COMMANDS.index('Circle')
    EOS_IDX = ALL_COMMANDS.index('EOS')
    SOL_IDX = ALL_COMMANDS.index('SOL')
    EXT_IDX = ALL_COMMANDS.index('Ext')

    print(f"Point Cloud path: {pc_path}")

    predicted_commands = list(seq_pred[:, 0])                 # (60)
    target_commands = list(seq_target[:, 0])
    
    predicted_seq_length = list(seq_target[:, 0]).index(EOS_IDX) + 3
    target_seq_length = list(seq_pred[:, 0]).index(EOS_IDX) + 3
    seq_length = target_seq_length if target_seq_length >= predicted_seq_length else predicted_seq_length

    predicted_commands = predicted_commands[:seq_length]
    target_commands = target_commands[:seq_length]

    predicted_command_list = [cmd for cmd in predicted_commands[:seq_length] for _ in range(cfg.n_args)] # (16 * seq_length)
    target_command_list = [cmd for cmd in target_commands[:seq_length] for _ in range(cfg.n_args)]       # (16 * seq_length)
    
    predicted_args = list(seq_pred[:seq_length, 1:])      # (seq_length, 16)
    target_args = list(seq_target[:seq_length, 1:])       # (seq_length, 16)
    
    predicted_cmd_args = torch.tensor([pred_arg for pred_cmd_args in predicted_args for pred_arg in pred_cmd_args]) # (16 * seq_length)
    target_cmd_args = torch.tensor([trgt_arg for trgt_cmd_args in target_args for trgt_arg in trgt_cmd_args])       # (16 * seq_length)

    print("pred")
    print(predicted_cmd_args[16], predicted_cmd_args[17])
    print("target")
    print(target_cmd_args[16], target_cmd_args[17])
    
    args_logits = args_logits[:seq_length]        # (seq_length, 16, 257)
    print("logits pred")
    print(args_logits[1,0,153])
    print(args_logits[1,1,88])
    args_softmax = softmax(args_logits)           # (seq_length, 16, 257)
    print("softmax pred")
    print(args_softmax[1,0,153])
    print(args_softmax[1,1,88])
            
    print(predicted_command_list)
    print(torch.arange(cfg.n_args).repeat(seq_length))
    predicted_cmd_args_sm = args_softmax[torch.repeat_interleave(torch.arange(seq_length), cfg.n_args), torch.arange(cfg.n_args).repeat(seq_length), predicted_cmd_args + 1]   # (seq_length * 16)
    target_cmd_args_sm = args_softmax[torch.repeat_interleave(torch.arange(seq_length), cfg.n_args), torch.arange(cfg.n_args).repeat(seq_length), target_cmd_args + 1]     # (seq_length * 16)

    print("softmax chosen")
    print(predicted_cmd_args_sm[16])
    print(predicted_cmd_args_sm[17])
    
    arg_loss = []
    counter = 0
    for i, cmd_args in enumerate(args_logits):
        for j, arg_logits in enumerate(cmd_args):
          #  print(counter)
           # print("argmax logits", np.argmax(arg_logits) - 1)
            #print("target_arg   ", target_cmd_args[counter].item())
       #     print("predicted arg", predicted_cmd_args[counter].item())
        #    print("predicted sm ", predicted_cmd_args_sm[counter].item())
         #   print("target    sm ", target_cmd_args_sm[counter].item())
          #  print(arg_logits[153])
           # print(arg_logits[88])
            #print("\n")
            arg_loss.append(cross_entropy(arg_logits, target_cmd_args[counter] + 1).item())
            counter += 1

    if cmd_idx != -1:
        target_command_list = target_command_list[cfg.n_args * cmd_idx:(cfg.n_args * cmd_idx) + cfg.n_args]
        predicted_command_list = predicted_command_list[cfg.n_args * cmd_idx:(cfg.n_args * cmd_idx) + cfg.n_args]
        target_cmd_args = target_cmd_args[cfg.n_args * cmd_idx:(cfg.n_args * cmd_idx) + cfg.n_args]
        predicted_cmd_args = predicted_cmd_args[cfg.n_args * cmd_idx:(cfg.n_args * cmd_idx) + cfg.n_args]
        target_cmd_args_sm = target_cmd_args_sm[cfg.n_args * cmd_idx:(cfg.n_args * cmd_idx) + cfg.n_args]
        predicted_cmd_args_sm = predicted_cmd_args_sm[cfg.n_args * cmd_idx:(cfg.n_args * cmd_idx) + cfg.n_args]
        arg_loss = arg_loss[cfg.n_args * cmd_idx:(cfg.n_args * cmd_idx) + cfg.n_args]
        
    df = pd.DataFrame(list(zip(target_command_list, 
                               predicted_command_list, 
                               target_cmd_args.tolist(), 
                               predicted_cmd_args.tolist(), 
                               [round(x * 100, 5)  for x in target_cmd_args_sm],
                               [round(x * 100, 5) for x in predicted_cmd_args_sm], 
                               arg_loss)),
                      columns=['trgt cmd', 'pred cmd', 'trgt', 'pred','prob_trgt', 'prob_pred', 'loss'])
    
    return df
        
show_results_args(0,-1) 



Point Cloud path: experiments/best_5_results/infered_point_clouds/00440420.ply
pred
tensor(152) tensor(87)
target
tensor(150) tensor(106)
logits pred
32.838802
33.9679
softmax pred
0.9998606
0.99621797
[4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3]
tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15,  0,  1,
         2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15,  0,  1,  2,  3,
         4,  

,trgt cmd,pred cmd,trgt,pred,prob_trgt,prob_pred,loss
0,4,4,-1,-1,0.00000,0.00000,25.552086
1,4,4,-1,-1,0.00000,0.00000,26.036608
2,4,4,-1,-1,0.00000,0.00000,33.243973
3,4,4,-1,-1,0.00006,0.00006,14.378495
4,4,4,-1,-1,0.00001,0.00001,16.373377
5,4,4,-1,-1,0.00008,0.00008,14.026071
6,4,4,-1,-1,0.00016,0.00016,13.344918
7,4,4,-1,-1,0.00008,0.00008,13.981358
8,4,4,-1,-1,0.00000,0.00000,25.233448
9,4,4,-1,-1,0.00000,0.00000,33.400623


In [146]:
torch.repeat_interleave(torch.arange(1, 4), 3)

tensor([1, 1, 1, 2, 2, 2, 3, 3, 3])

In [148]:
torch.aseqrange(4)

tensor([0, 1, 2, 3])

In [145]:
abc = torch.tensor([-1.1593e+01, -1.1793e+01, -1.1749e+01, -1.1726e+01, -1.2273e+01,
        -1.1608e+01, -1.1866e+01, -1.1998e+01, -1.1636e+01, -1.2013e+01,
        -1.1954e+01, -1.1791e+01, -1.1933e+01, -1.1549e+01, -1.1704e+01,
        -1.2033e+01, -1.1244e+01, -1.1987e+01, -1.1347e+01, -1.1576e+01,
        -1.1921e+01, -1.1921e+01, -1.1136e+01, -1.1712e+01, -1.2110e+01,
        -1.1784e+01, -1.2137e+01, -1.1776e+01, -1.1691e+01, -1.1797e+01,
        -1.2182e+01, -1.1932e+01, -1.1905e+01, -1.1734e+01, -1.1973e+01,
        -1.2023e+01, -1.2118e+01, -2.3991e+01, -1.8785e+01, -1.1634e+01,
        -1.2095e+01, -1.1776e+01, -1.2037e+01, -1.2041e+01, -1.1660e+01,
        -2.7247e+01, -3.2370e+01, -3.6731e+01, -1.4870e+01, -1.1775e+01,
        -3.2772e+01, -2.5371e+01, -1.8860e+01, -3.5922e+01, -1.1875e+01,
        -1.8692e+01, -4.6048e+01, -3.2303e+01, -1.1452e+01, -1.1416e+01,
        -1.1846e+01, -2.4532e+01, -2.0030e+01, -3.1591e+01, -1.1655e+01,
        -3.6347e+01, -4.7404e+01, -4.2596e+01, -3.6891e+01, -1.9054e+01,
        -1.1956e+01, -2.7473e+01, -4.0038e+01, -1.1881e+01, -2.3622e+01,
        -2.8459e+01, -3.5045e+01, -4.9315e+01, -2.9904e+01, -2.7833e+01,
        -3.5874e+01, -2.7981e+01, -3.5803e+01, -3.4442e+01, -3.7835e+01,
        -3.8357e+01, -4.0295e+01, -2.9129e+01, -4.9995e+01, -2.8963e+01,
        -3.7292e+01, -3.3842e+01, -4.0183e+01, -3.3794e+01, -3.5494e+01,
        -4.1179e+01, -1.6192e+01, -4.3450e+01, -2.0802e+01, -3.1672e+01,
        -3.4327e+01, -2.9905e+01, -3.3556e+01, -3.7029e+01, -4.0676e+01,
        -3.5521e+01, -4.6741e+01, -3.7923e+01, -2.8641e+01, -3.3149e+01,
        -2.7566e+01, -3.6125e+01, -3.9658e+01, -4.1691e+01, -2.5125e+01,
        -3.9021e+01, -3.4893e+01, -4.3622e+01, -2.9656e+01, -3.3523e+01,
        -3.4969e+01, -1.8396e+01, -4.8976e+01, -3.7206e+01, -4.1237e+01,
        -3.3821e+01, -3.1416e+01, -4.5186e+01, -2.3094e+01,  1.9526e+01,
        -5.8446e+00, -1.6600e+01, -1.3630e+01, -6.4358e+00, -8.1660e+00,
        -1.9094e+01, -1.5312e+01, -1.1254e+01, -8.0631e+00, -1.5693e+00,
        -7.3584e+00, -1.5154e+01, -2.1116e+01, -1.3964e+01, -2.0031e+01,
        -1.7515e+01, -1.8320e+01, -1.9534e+01, -4.3450e+00, -9.8405e+00,
        -1.4515e+01, -2.1408e+01, -1.9525e+01, -8.7235e+00, -2.2997e+01,
        -1.1529e+01, -1.3846e+01, -2.0147e+00, -5.3765e+00, -1.1498e+01,
        -1.4293e+01, -2.9460e+00, -1.5774e+01, -1.2226e+01, -2.2472e+01,
        -1.5589e+01, -1.1127e+01,  2.5581e+00, -8.7392e+00, -5.5238e+00,
        -6.6651e+00, -3.9901e+00, -6.3465e+00, -1.6003e+01, -1.1131e+01,
        -6.4326e+00, -3.0762e+00,  1.9682e+01,  1.4941e+00, -4.8983e+00,
        -2.8226e+00,  3.3351e-01,  8.8367e+00,  4.6910e+00,  1.0023e+01,
         1.2569e+00,  8.0403e+00,  1.8695e-01, -6.0215e+00, -5.2542e+00,
        -4.5150e+00,  1.9806e-02,  1.1289e+01,  1.7565e+00,  2.7028e-01,
         1.3246e+01,  7.6768e+00,  7.6479e+00,  4.9198e+00, -5.7551e+00,
         9.2847e+00, -5.0201e+00, -4.7274e+00,  7.7945e+00,  1.3155e+01,
         1.7878e+01, -3.1058e+00, -1.9038e+00,  3.2455e+00,  5.6601e+00,
         6.4752e+00,  5.9353e+00,  7.7235e+00,  1.4824e+01,  1.2250e+01,
         1.5599e+01,  5.6602e+00,  6.2073e+00,  5.8759e+00,  1.8519e+01,
         1.7664e+01,  1.2346e+01,  9.3626e+00,  1.9000e+01,  5.2011e+01,
        -1.1926e+01, -1.1710e+01, -1.1996e+01, -1.1809e+01, -1.1659e+01,
        -1.1614e+01, -1.1408e+01, -1.1309e+01, -1.1426e+01, -1.1537e+01,
        -1.1177e+01, -1.1863e+01, -1.1366e+01, -1.1675e+01, -1.2009e+01,
        -1.1899e+01, -1.1994e+01, -1.1057e+01, -1.1955e+01, -1.1881e+01,
        -1.1560e+01, -1.1984e+01, -1.1641e+01, -1.1752e+01, -1.1622e+01,
        -1.2267e+01, -1.1988e+01, -1.2264e+01, -1.2313e+01, -1.1675e+01,
        -1.1903e+01, -1.1663e+01])

In [147]:
torch.argmax(abc)
torch.sum(abc)

tensor(-3778.5574)

In [26]:
show_results_args(0, 1)

(60, 16, 257) (60, 6) (60, 17) (60, 17) (256,) (256,)
Point Cloud path: experiments/best_5_results/infered_point_clouds/00575185.ply
(60, 16, 257)
[204 128  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1]
[206 128  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1]
[[  4.936258   -11.334982   -12.412564   -21.583628    -4.397084
  -14.8361     -16.388008   -16.743713    -0.03594986 -12.276738
   -9.990585    -6.9201374   -7.9274597    1.7885818  -17.452883
  -17.203762  ]
 [-34.934937    17.211254     3.4887717  -21.455511    -4.319825
  -14.038717   -16.694796   -16.840965     9.060187     5.083467
   10.411677    -8.987089    -5.594931    -8.011899   -17.759392
  -16.623045  ]
 [-16.667828   -13.389228   -20.159206   -21.528215    -4.7289324
   -0.30675676 -15.744706    -3.0173824   -3.5703645  -20.636581
  -20.985083    -8.817478   -10.37203     -3.1957917  -18.198559
  -17.071405  ]
 [-16.667828   -13.389228   -20.159206   -21.528215    -4.7289324
   -0.30675676 -15.74

In [ ]:
export2step()

In [22]:
step2stl()

### Gedanken

Was noch wichtig/Meeting morgen:

- args loss visualization

Meeting: 
- had to finish applications
- first thing I did was refactor training
    - Automatic resume of training if cluster fails
    - Parallelization (30mins/epoch -> 12 mins/epoch)
- implemented test script
- Implemented cosine annealing learning rate -> show new training with better convergence!
- Worked on CAD Loss understanding
- Implemented testing pipeline to make sure the data is alligned
- Finished pc2cad pipeline with thorough understanding of loss for train/val/test
- Created interactive pc2cad

HiWi:
- created SAiL poster
- documented literature research
- made Blensor work

Next:

- train DeepCAD ourselves?
- Train both models in one pipeline using CADLoss?
- use blensor to create new data?
